# Merge SPY and EEM holdings files

This notebook:
1. Mounts Google Drive
2. Reads all yearly holdings `.xlsx` files from the two folders
3. Standardises columns
4. Combines them into:
   - one merged **SPY** file
   - one merged **EEM** file
5. Saves the merged outputs back to Google Drive

Expected folders:
- `/content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/spy data`
- `/content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/eem data`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import re
import glob
import pandas as pd

SPY_DIR = "/content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/etf spy data"
EEM_DIR = "/content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/etf eem data"

OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/merged_holdings"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SPY_OUTPUT = os.path.join(OUTPUT_DIR, "SPY_holdings_2015_2022_merged.xlsx")
EEM_OUTPUT = os.path.join(OUTPUT_DIR, "EEM_holdings_2015_2022_merged.xlsx")

print("SPY_DIR:", SPY_DIR)
print("EEM_DIR:", EEM_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


SPY_DIR: /content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/etf spy data
EEM_DIR: /content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/etf eem data
OUTPUT_DIR: /content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/merged_holdings


In [ ]:
def extract_year_from_filename(filepath: str):
    name = os.path.basename(filepath)
    match = re.search(r"(20\d{2})", name)
    return int(match.group(1)) if match else None

def clean_weight_column(series: pd.Series) -> pd.Series:
    # Handles values like '0.04%' or numeric values
    as_str = (
        series.astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    return pd.to_numeric(as_str, errors="coerce")

def load_and_standardise_one_file(filepath: str, etf_name: str) -> pd.DataFrame:
    df = pd.read_excel(filepath)
    df.columns = [str(c).strip() for c in df.columns]

    wanted = ["RIC", "Name", "Country", "date", "Weight", "No. Shares", "Change"]
    existing = [c for c in wanted if c in df.columns]
    df = df[existing].copy()

    for col in wanted:
        if col not in df.columns:
            df[col] = pd.NA

    df = df.dropna(how="all")

    df["RIC"] = df["RIC"].astype(str).str.strip()
    df["Name"] = df["Name"].astype(str).str.strip()
    df["Country"] = df["Country"].astype(str).str.strip()
    df["Weight"] = clean_weight_column(df["Weight"])
    df["No. Shares"] = pd.to_numeric(
        df["No. Shares"].astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce"
    )
    df["Change"] = pd.to_numeric(
        df["Change"].astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce"
    )

    df["source_file"] = os.path.basename(filepath)
    df["etf"] = etf_name
    # df["year"] = extract_year_from_filename(filepath)

    ordered_cols = [
        "etf", "date", "RIC", "Name", "Country", "Weight",
        "No. Shares", "Change", "source_file"
    ]
    df = df[ordered_cols]

    df = df[df["Name"].notna()]
    df = df[df["Name"].astype(str).str.strip() != ""]
    df = df[df["Weight"].notna()]

    return df

def merge_holdings_folder(folder_path: str, etf_name: str) -> pd.DataFrame:
    files = sorted(glob.glob(os.path.join(folder_path, "*.xlsx")))
    if not files:
        raise FileNotFoundError(f"No .xlsx files found in: {folder_path}")

    merged = []
    for f in files:
        one = load_and_standardise_one_file(f, etf_name=etf_name)
        merged.append(one)

    out = pd.concat(merged, ignore_index=True)

    # Date processing
    out['date'] = pd.to_datetime(out['date'], errors='coerce')

    # Check for corrupted dates
    print(f"{etf_name} Number of dates that cannot be converte：", out['date'].isna().sum())

    # Sort (use datetime)
    out = out.sort_values(["date", "Weight"], ascending=[True, False]).reset_index(drop=True)

    return out

spy_merged = merge_holdings_folder(SPY_DIR, "SPY")
eem_merged = merge_holdings_folder(EEM_DIR, "EEM")

print("SPY merged shape:", spy_merged.shape)
print("EEM merged shape:", eem_merged.shape)


In [ ]:
display(spy_merged.head())
display(eem_merged.head())

print("SPY years:", sorted(spy_merged["date"].dropna().unique().tolist()))
print("EEM years:", sorted(eem_merged["date"].dropna().unique().tolist()))

print("\nSPY total weight by date:")
print(spy_merged.groupby("date")["Weight"].sum())

print("\nEEM total weight by date:")
print(eem_merged.groupby("date")["Weight"].sum())


,etf,date,RIC,Name,Country,Weight,No. Shares,Change,source_file,RIC_missing
0,SPY,2015-12-31,AAPL.OQ,APPLE INC ORD,UNITED STATES,0.032670,56501081.0,571045.0,Derived Holdings 2026-04-13 SPY 2015.xlsx,False
1,SPY,2015-12-31,MSFT.OQ,MICROSOFT CORP ORD,UNITED STATES,0.024681,80982372.0,2541213.0,Derived Holdings 2026-04-13 SPY 2015.xlsx,False
2,SPY,2015-12-31,XOM.N,EXXON MOBIL CORP ORD,UNITED STATES,0.018072,42204340.0,1318973.0,Derived Holdings 2026-04-13 SPY 2015.xlsx,False
3,SPY,2015-12-31,GE.N,GENERAL ELECTRIC CO ORD,UNITED STATES,0.016374,95686751.0,3309174.0,Derived Holdings 2026-04-13 SPY 2015.xlsx,False
4,SPY,2015-12-31,JNJ.N,JOHNSON & JOHNSON ORD,UNITED STATES,0.015829,28051475.0,893143.0,Derived Holdings 2026-04-13 SPY 2015.xlsx,False


,etf,date,RIC,Name,Country,Weight,No. Shares,Change,source_file,RIC_missing
0,EEM,2015-12-31,005930.KS,SAMSUNG ELECTRONICS CO LTD ORD,KOREA,0.034278,686788.0,-29087.0,Derived Holdings 2026-04-13 EEM 2015.xlsx,False
1,EEM,2015-12-31,2330.TW,TAIWAN SEMICONDUCTOR MANUFACTURING CO LTD ORD,TAIWAN,0.030986,153236000.0,-6395000.0,Derived Holdings 2026-04-13 EEM 2015.xlsx,False
2,EEM,2015-12-31,0700.HK,TENCENT HOLDINGS LTD ORD,CHINA,0.029424,32194800.0,-1201900.0,Derived Holdings 2026-04-13 EEM 2015.xlsx,False
3,EEM,2015-12-31,0941.HK,CHINA MOBILE LTD ORD,CHINA,0.020067,38267000.0,-1600500.0,Derived Holdings 2026-04-13 EEM 2015.xlsx,False
4,EEM,2015-12-31,0939.HK,CHINA CONSTRUCTION BANK CORP ORD,CHINA,0.016666,523707760.0,-21372000.0,Derived Holdings 2026-04-13 EEM 2015.xlsx,False


SPY years: [Timestamp('2015-12-31 00:00:00'), Timestamp('2016-12-31 00:00:00'), Timestamp('2017-12-31 00:00:00'), Timestamp('2018-12-31 00:00:00'), Timestamp('2019-12-31 00:00:00'), Timestamp('2020-12-31 00:00:00'), Timestamp('2021-12-31 00:00:00'), Timestamp('2022-12-31 00:00:00')]
EEM years: [Timestamp('2015-12-31 00:00:00'), Timestamp('2016-12-31 00:00:00'), Timestamp('2017-12-31 00:00:00'), Timestamp('2018-12-31 00:00:00'), Timestamp('2019-12-31 00:00:00'), Timestamp('2020-12-31 00:00:00'), Timestamp('2021-12-31 00:00:00'), Timestamp('2022-12-31 00:00:00')]

SPY total weight by date:
date
2015-12-31    1.002206
2016-12-31    1.006137
2017-12-31    0.981186
2018-12-31    0.991574
2019-12-31    1.018777
2020-12-31    1.013663
2021-12-31    1.000000
2022-12-31    0.999998
Name: Weight, dtype: float64

EEM total weight by date:
date
2015-12-31    1.000006
2016-12-31    1.002269
2017-12-31    0.999999
2018-12-31    0.999997
2019-12-31    0.999999
2020-12-31    1.000003
2021-12-31    1.0

In [ ]:
spy_merged.head()

,etf,date,RIC,Name,Country,Weight,No. Shares,Change,source_file,RIC_missing
0,SPY,2015-12-31,AAPL.OQ,APPLE INC ORD,UNITED STATES,0.032670,56501081.0,571045.0,Derived Holdings 2026-04-13 SPY 2015.xlsx,False
1,SPY,2015-12-31,MSFT.OQ,MICROSOFT CORP ORD,UNITED STATES,0.024681,80982372.0,2541213.0,Derived Holdings 2026-04-13 SPY 2015.xlsx,False
2,SPY,2015-12-31,XOM.N,EXXON MOBIL CORP ORD,UNITED STATES,0.018072,42204340.0,1318973.0,Derived Holdings 2026-04-13 SPY 2015.xlsx,False
3,SPY,2015-12-31,GE.N,GENERAL ELECTRIC CO ORD,UNITED STATES,0.016374,95686751.0,3309174.0,Derived Holdings 2026-04-13 SPY 2015.xlsx,False
4,SPY,2015-12-31,JNJ.N,JOHNSON & JOHNSON ORD,UNITED STATES,0.015829,28051475.0,893143.0,Derived Holdings 2026-04-13 SPY 2015.xlsx,False


In [ ]:
with pd.ExcelWriter(SPY_OUTPUT, engine="openpyxl") as writer:
    spy_merged.to_excel(writer, index=False, sheet_name="SPY_holdings")

with pd.ExcelWriter(EEM_OUTPUT, engine="openpyxl") as writer:
    eem_merged.to_excel(writer, index=False, sheet_name="EEM_holdings")

print("Saved:")
print(SPY_OUTPUT)
print(EEM_OUTPUT)


Saved:
/content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/merged_holdings/SPY_holdings_2015_2022_merged.xlsx
/content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/merged_holdings/EEM_holdings_2015_2022_merged.xlsx


In [ ]:
spy_csv = SPY_OUTPUT.replace(".xlsx", ".csv")
eem_csv = EEM_OUTPUT.replace(".xlsx", ".csv")

spy_merged.to_csv(spy_csv, index=False)
eem_merged.to_csv(eem_csv, index=False)

print("Also saved CSV files:")
print(spy_csv)
print(eem_csv)


Also saved CSV files:
/content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/merged_holdings/SPY_holdings_2015_2022_merged.csv
/content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/merged_holdings/EEM_holdings_2015_2022_merged.csv


The next section is:

* Read in SPY_ESG and EEM_ESG

* Change the text "NULL" in the fields to a true null value

* Create a year using Financial Period Absolute

* Read in SPY_holdings_2015_2022_merged_clean_v2 and EEM_holdings_2015_2022_merged_clean_v2

* Create a year using date

* Merge the ESG back to holdings using RIC + year

* Save as a new merged file to the merged_holdings folder

In [ ]:
import os
import re
import pandas as pd

# =========================
# 1. path setting
# =========================
spy_esg_path = "/content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/etf spy data/LSEG_ESG_SPY_2015_2024.xlsx"
eem_esg_path = "/content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/etf eem data/LSEG_ESG_EEM_2015_2024.xlsx"

spy_holdings_path = "/content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/merged_holdings/SPY_holdings_2015_2022_merged_clean_v2.xlsx"
eem_holdings_path = "/content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/merged_holdings/EEM_holdings_2015_2022_merged_clean_v2.xlsx"

output_dir = "/content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/merged_holdings"

spy_output_path = os.path.join(output_dir, "SPY_holdings_2015_2022_merged_with_ESG.xlsx")
eem_output_path = os.path.join(output_dir, "EEM_holdings_2015_2022_merged_with_ESG.xlsx")


# =========================
# 2. function
# =========================
def extract_year_from_financial_period(value):
    """
    create "year" from "Financial Period Absolute"
    example:
    FY2014 -> 2014
    FY2022 -> 2022
    """
    if pd.isna(value):
        return pd.NA

    match = re.search(r"(20\d{2})", str(value))
    return int(match.group(1)) if match else pd.NA


def clean_esg_file(esg_path):
    """
    讀 ESG 檔案，做：
    1. 'NULL' -> 空值
    2. 建立 year
    3. 保留 merge 需要欄位
    """
    esg = pd.read_excel(esg_path)
    esg.columns = [str(c).strip() for c in esg.columns]

    # Convert NULL text to a true null value
    esg = esg.replace("NULL", pd.NA)

    # Create year
    esg["year"] = esg["Financial Period Absolute"].apply(extract_year_from_financial_period)

    # If there are duplicate RIC + year entries, only keep the first one
    esg = esg.drop_duplicates(subset=["RIC", "year"]).reset_index(drop=True)

    return esg


def clean_holdings_file(holdings_path):
    """
    Read the holdings file and create year from date
    """
    holdings = pd.read_excel(holdings_path)
    holdings.columns = [str(c).strip() for c in holdings.columns]

    holdings["date"] = pd.to_datetime(holdings["date"], errors="coerce")
    holdings["year"] = holdings["date"].dt.year

    return holdings


def merge_holdings_with_esg(holdings_df, esg_df):
    """
    Use RIC + year to pick up ESG
    """
    merged = holdings_df.merge(
        esg_df,
        left_on=["RIC", "year"],
        right_on=["RIC", "year"],
        how="left",
        suffixes=("", "_esg")
    )
    return merged


# =========================
# 3. read ESG data
# =========================
spy_esg = clean_esg_file(spy_esg_path)
eem_esg = clean_esg_file(eem_esg_path)

print("SPY ESG shape:", spy_esg.shape)
print("EEM ESG shape:", eem_esg.shape)


# =========================
# 4. read holdings data and sort
# =========================
spy_holdings = clean_holdings_file(spy_holdings_path)
eem_holdings = clean_holdings_file(eem_holdings_path)

print("SPY holdings shape:", spy_holdings.shape)
print("EEM holdings shape:", eem_holdings.shape)


# =========================
# 5. merge ESG
# note：with RIC + year
# =========================
spy_merged_esg = merge_holdings_with_esg(spy_holdings, spy_esg)
eem_merged_esg = merge_holdings_with_esg(eem_holdings, eem_esg)

print("SPY merged with ESG shape:", spy_merged_esg.shape)
print("EEM merged with ESG shape:", eem_merged_esg.shape)


# =========================
# 6. save results
# =========================
spy_merged_esg.to_excel(spy_output_path, index=False)
eem_merged_esg.to_excel(eem_output_path, index=False)

print(f"Saved: {spy_output_path}")
print(f"Saved: {eem_output_path}")

SPY ESG shape: (6000, 8)
EEM ESG shape: (14455, 8)
SPY holdings shape: (4038, 9)
EEM holdings shape: (8337, 9)
SPY merged with ESG shape: (4038, 15)
EEM merged with ESG shape: (8337, 15)
Saved: /content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/merged_holdings/SPY_holdings_2015_2022_merged_with_ESG.xlsx
Saved: /content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/merged_holdings/EEM_holdings_2015_2022_merged_with_ESG.xlsx


Section 0: First, define the path and shared functions.

In [ ]:
import os
import re
import pandas as pd

# =========================
# path setting
# =========================
MERGED_DIR = "/content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/merged_holdings"
SPY_DIR = "/content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/etf spy data"
EEM_DIR = "/content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/etf eem data"
CLEAN_DIR = "/content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/clean_data"

os.makedirs(CLEAN_DIR, exist_ok=True)

# =========================
# file name setting
# =========================
SPY_HOLDINGS_V2 = "SPY_holdings_2015_2022_merged_clean_v2"
EEM_HOLDINGS_V2 = "EEM_holdings_2015_2022_merged_clean_v2"
RIC_CORRECTION_FILE = "LSEG_RIC_correction"

SPY_ESG_V1 = "LSEG_ESG_SPY_2015_2024"
EEM_ESG_V1 = "LSEG_ESG_EEM_2015_2024"

SPY_HOLDINGS_V3 = "SPY_holdings_2015_2022_merged_clean_v3.xlsx"
EEM_HOLDINGS_V3 = "EEM_holdings_2015_2022_merged_clean_v3.xlsx"

SPY_ESG_V2 = "LSEG_ESG_SPY_2015_2024_v2.xlsx"
EEM_ESG_V2 = "LSEG_ESG_EEM_2015_2024_v2.xlsx"

SPY_MERGED_ESG = "SPY_holdings_2015_2022_merged_with_ESG.xlsx"
EEM_MERGED_ESG = "EEM_holdings_2015_2022_merged_with_ESG.xlsx"


# =========================
# common function
# =========================
def find_file(base_dir, file_stem):
    """
    Automatically finds xlsx/csv files
    """
    for ext in [".xlsx", ".csv"]:
        path = os.path.join(base_dir, file_stem + ext)
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"File not found: {os.path.join(base_dir, file_stem)} (.xlsx or .csv)")


def read_table(path):
    """
    Automatically reads xlsx/csv files
    """
    if path.lower().endswith(".xlsx"):
        df = pd.read_excel(path)
    elif path.lower().endswith(".csv"):
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file format：{path}")
    df.columns = [str(c).strip() for c in df.columns]
    return df


def save_xlsx(df, path):
    df.to_excel(path, index=False)
    print(f"Saved: {path}")


def extract_year_from_fy(value):
    """
    FY2015 -> 2015
    """
    if pd.isna(value):
        return pd.NA
    m = re.search(r"(20\d{2})", str(value))
    return int(m.group(1)) if m else pd.NA


def extract_year_from_date(value):
    """
    2020-12-31 / 2020-12-31 00:00:00 -> 2020
    """
    dt = pd.to_datetime(value, errors="coerce")
    return dt.year if pd.notna(dt) else pd.NA


def normalize_text(s):
    """
    Used to detect if the name only differs from ORD/PFD
    """
    if pd.isna(s):
        return ""
    s = str(s).upper().strip()

    # Remove common slashes
    remove_patterns = [
        r"\bORDINARY SHARE\b",
        r"\bORD SHS\b",
        r"\bORD\b",
        r"\bPREFERRED SHARE\b",
        r"\bPREF(?:ERENCE)?\b",
        r"\bPFD\b",
        r"\bPRF\b",
        r"\bDR\b",
        r"\bADR\b",
        r"\bREP\b",
        r"\bCL(?:ASS)?\s*[A-Z0-9]+\b",
        r"\bNON[- ]?VOTING\b",
    ]
    for p in remove_patterns:
        s = re.sub(p, "", s)

    # Remove redundant symbols and whitespace
    s = re.sub(r"[^A-Z0-9 ]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


print("path and function have saved")

Section 1: Modify the RIC of holdings, create old_RIC and year, and save it as v3.

In [ ]:
# =========================
# Paragraph 1: holdings v2 -> v3
# =========================

# read file
spy_holdings_path = find_file(MERGED_DIR, SPY_HOLDINGS_V2)
eem_holdings_path = find_file(MERGED_DIR, EEM_HOLDINGS_V2)
ric_corr_path = find_file(MERGED_DIR, RIC_CORRECTION_FILE)

spy_holdings = read_table(spy_holdings_path)
eem_holdings = read_table(eem_holdings_path)
ric_corr = read_table(ric_corr_path)

print("SPY holdings v2 shape:", spy_holdings.shape)
print("EEM holdings v2 shape:", eem_holdings.shape)
print("RIC correction shape:", ric_corr.shape)

print("\nRIC correction columns:")
print(ric_corr.columns.tolist())

# correction table
ric_corr = ric_corr.copy()
ric_corr.columns = [str(c).strip() for c in ric_corr.columns]

required_corr_cols = ["ETF", "RIC", "RIC_correct"]
missing_corr_cols = [c for c in required_corr_cols if c not in ric_corr.columns]
if missing_corr_cols:
    raise KeyError(f"LSEG_RIC_correction 缺少欄位：{missing_corr_cols}")

ric_corr["ETF"] = ric_corr["ETF"].astype(str).str.strip().str.upper()
ric_corr["RIC"] = ric_corr["RIC"].astype(str).str.strip()
ric_corr["RIC_correct"] = ric_corr["RIC_correct"].astype(str).str.strip()

ric_corr = ric_corr.drop_duplicates(subset=["ETF", "RIC"]).reset_index(drop=True)

print("\nRIC correction preview:")
print(ric_corr.head())


def apply_ric_correction(holdings_df, etf_name, corr_df):
    df = holdings_df.copy()
    df["etf"] = df["etf"].astype(str).str.strip().str.upper()

    # keep old RIC
    df["old_RIC"] = df["RIC"]

    # year
    df["year"] = df["date"].apply(extract_year_from_date)

    # ETF correction
    corr_sub = corr_df[corr_df["ETF"] == etf_name].copy()

    # merge
    df = df.merge(
        corr_sub[["RIC", "RIC_correct"]],
        on="RIC",
        how="left"
    )

    # Replace if a correction value is available; otherwise, keep the original value.
    df["RIC"] = df["RIC_correct"].combine_first(df["RIC"])
    df = df.drop(columns=["RIC_correct"])

    return df


spy_v3 = apply_ric_correction(spy_holdings, "SPY", ric_corr)
eem_v3 = apply_ric_correction(eem_holdings, "EEM", ric_corr)

print("\nSPY v3 preview:")
print(spy_v3.head())

print("\nEEM v3 preview:")
print(eem_v3.head())

# Check how many entries were corrected
spy_changed = (spy_v3["old_RIC"].astype(str) != spy_v3["RIC"].astype(str)).sum()
eem_changed = (eem_v3["old_RIC"].astype(str) != eem_v3["RIC"].astype(str)).sum()

print(f"\nSPY corrected RIC rows: {spy_changed}")
print(f"EEM corrected RIC rows: {eem_changed}")

# save results
spy_v3_path = os.path.join(MERGED_DIR, SPY_HOLDINGS_V3)
eem_v3_path = os.path.join(MERGED_DIR, EEM_HOLDINGS_V3)

save_xlsx(spy_v3, spy_v3_path)
save_xlsx(eem_v3, eem_v3_path)

SPY holdings v2 shape: (4038, 7)
EEM holdings v2 shape: (8335, 7)
RIC correction shape: (48, 3)

RIC correction columns:
['ETF', 'RIC', 'RIC_correct']

RIC correction preview:
   ETF     RIC RIC_correct
0  SPY   AET.N   ARG.N^E16
1  SPY   ARG.N   AET.N^K18
2  SPY  BXLT.N  BXLT.N^F16
3  SPY   BCR.N   BCR.N^L17
4  SPY   CVC.N   CVC.N^F16

SPY v3 preview:
   etf        date    RIC       Name        Country    Weight  No. Shares  \
0  SPY  2015-12-31  MMM.N  3M CO ORD  UNITED STATES  0.005164     6240677   
1  SPY  2016-12-31  MMM.N  3M CO ORD  UNITED STATES  0.005586     7038215   
2  SPY  2017-12-31  MMM.N  3M CO ORD  UNITED STATES  0.006003     7078439   
3  SPY  2018-12-31  MMM.N  3M CO ORD  UNITED STATES  0.005205     6649048   
4  SPY  2019-12-31  MMM.N  3M CO ORD  UNITED STATES  0.003840     6691310   

  old_RIC  year  
0   MMM.N  2015  
1   MMM.N  2016  
2   MMM.N  2017  
3   MMM.N  2018  
4   MMM.N  2019  

EEM v3 preview:
   etf        date        RIC  \
0  EEM  2019-12-31  6013

Section 2: Organize the ESG, create a year, and save as v2.

In [ ]:
# =========================
# secition 2：ESG v1 -> v2
# =========================

spy_esg_path = find_file(SPY_DIR, SPY_ESG_V1)
eem_esg_path = find_file(EEM_DIR, EEM_ESG_V1)

spy_esg = read_table(spy_esg_path)
eem_esg = read_table(eem_esg_path)

print("SPY ESG shape:", spy_esg.shape)
print("EEM ESG shape:", eem_esg.shape)

print("\nSPY ESG columns:")
print(spy_esg.columns.tolist())

print("\nEEM ESG columns:")
print(eem_esg.columns.tolist())


def clean_esg(df):
    esg = df.copy()

    # covert"NULL" to NA
    esg = esg.replace("NULL", pd.NA)

    # check column name
    if "Financial Period Absolute" not in esg.columns:
        raise KeyError("ESG missing column：Financial Period Absolute")
    if "RIC" not in esg.columns:
        raise KeyError("ESG missing column：RIC")

    # year
    esg["year"] = esg["Financial Period Absolute"].apply(extract_year_from_fy)

    # sorting
    esg["RIC"] = esg["RIC"].astype(str).str.strip()
    esg.loc[esg["RIC"].str.lower() == "nan", "RIC"] = pd.NA
    esg.loc[esg["RIC"] == "", "RIC"] = pd.NA

    esg = esg.drop_duplicates(subset=["RIC", "year"]).reset_index(drop=True)

    return esg


spy_esg_v2 = clean_esg(spy_esg)
eem_esg_v2 = clean_esg(eem_esg)

print("\nSPY ESG v2 preview:")
print(spy_esg_v2.head())

print("\nEEM ESG v2 preview:")
print(eem_esg_v2.head())

# save results
spy_esg_v2_path = os.path.join(SPY_DIR, SPY_ESG_V2)
eem_esg_v2_path = os.path.join(EEM_DIR, EEM_ESG_V2)

save_xlsx(spy_esg_v2, spy_esg_v2_path)
save_xlsx(eem_esg_v2, eem_esg_v2_path)

SPY ESG shape: (6107, 9)
EEM ESG shape: (14728, 9)

SPY ESG columns:
['RIC', 'Date', 'Financial Period Absolute', 'Calc Date', 'ESG Score', 'Environmental Pillar Score', 'Social Pillar Score', 'Governance Pillar Score', 'ESG Combined Score Grade']

EEM ESG columns:
['RIC', 'Date', 'Financial Period Absolute', 'Calc Date', 'ESG Score', 'Environmental Pillar Score', 'Social Pillar Score', 'Governance Pillar Score', 'ESG Combined Score Grade']

SPY ESG v2 preview:
     RIC                 Date Financial Period Absolute            Calc Date  \
0  MMM.N  2014-12-31 00:00:00                    FY2014  2015-12-31 00:00:00   
1  MMM.N  2015-12-31 00:00:00                    FY2015  2016-12-31 00:00:00   
2  MMM.N  2016-12-31 00:00:00                    FY2016  2017-12-31 00:00:00   
3  MMM.N  2017-12-31 00:00:00                    FY2017  2018-12-31 00:00:00   
4  MMM.N  2018-12-31 00:00:00                    FY2018  2019-12-31 00:00:00   

   ESG Score  Environmental Pillar Score  Social Pill

第 3 段：用 RIC + year merge ESG 回 holdings，存 merged_with_ESG

In [ ]:
# =========================
# section 3：holdings v3 merge ESG v2
# =========================

# Reread the clean version to avoid confusion when continuing from the old notebook.
spy_v3 = read_table(os.path.join(MERGED_DIR, SPY_HOLDINGS_V3))
eem_v3 = read_table(os.path.join(MERGED_DIR, EEM_HOLDINGS_V3))

spy_esg_v2 = read_table(os.path.join(SPY_DIR, SPY_ESG_V2))
eem_esg_v2 = read_table(os.path.join(EEM_DIR, EEM_ESG_V2))

print("SPY holdings v3:", spy_v3.shape)
print("EEM holdings v3:", eem_v3.shape)
print("SPY ESG v2:", spy_esg_v2.shape)
print("EEM ESG v2:", eem_esg_v2.shape)


def merge_holdings_esg(holdings_df, esg_df):
    h = holdings_df.copy()
    e = esg_df.copy()

    # year formatted into a mergeable structure
    h["year"] = pd.to_numeric(h["year"], errors="coerce")
    e["year"] = pd.to_numeric(e["year"], errors="coerce")

    h["RIC"] = h["RIC"].astype(str).str.strip()
    e["RIC"] = e["RIC"].astype(str).str.strip()

    merged = h.merge(
        e,
        on=["RIC", "year"],
        how="left",
        suffixes=("", "_esg")
    )
    return merged


spy_with_esg = merge_holdings_esg(spy_v3, spy_esg_v2)
eem_with_esg = merge_holdings_esg(eem_v3, eem_esg_v2)

print("\nSPY merged preview:")
print(spy_with_esg.head())

print("\nEEM merged preview:")
print(eem_with_esg.head())

# You can use these fields to view the merge success rate
for col in ["ESG Score", "Environmental Pillar Score", "Social Pillar Score", "Governance Pillar Score"]:
    if col in spy_with_esg.columns:
        print(f"\nSPY match rate by '{col}':", spy_with_esg[col].notna().mean())
        break

for col in ["ESG Score", "Environmental Pillar Score", "Social Pillar Score", "Governance Pillar Score"]:
    if col in eem_with_esg.columns:
        print(f"EEM match rate by '{col}':", eem_with_esg[col].notna().mean())
        break

# save results
spy_with_esg_path = os.path.join(MERGED_DIR, SPY_MERGED_ESG)
eem_with_esg_path = os.path.join(MERGED_DIR, EEM_MERGED_ESG)

save_xlsx(spy_with_esg, spy_with_esg_path)
save_xlsx(eem_with_esg, eem_with_esg_path)

SPY holdings v3: (4038, 9)
EEM holdings v3: (8335, 9)
SPY ESG v2: (6104, 10)
EEM ESG v2: (14702, 10)

SPY merged preview:
   etf        date    RIC       Name        Country    Weight  No. Shares  \
0  SPY  2015-12-31  MMM.N  3M CO ORD  UNITED STATES  0.005164     6240677   
1  SPY  2016-12-31  MMM.N  3M CO ORD  UNITED STATES  0.005586     7038215   
2  SPY  2017-12-31  MMM.N  3M CO ORD  UNITED STATES  0.006003     7078439   
3  SPY  2018-12-31  MMM.N  3M CO ORD  UNITED STATES  0.005205     6649048   
4  SPY  2019-12-31  MMM.N  3M CO ORD  UNITED STATES  0.003840     6691310   

  old_RIC  year                 Date Financial Period Absolute  \
0   MMM.N  2015  2015-12-31 00:00:00                    FY2015   
1   MMM.N  2016  2016-12-31 00:00:00                    FY2016   
2   MMM.N  2017  2017-12-31 00:00:00                    FY2017   
3   MMM.N  2018  2018-12-31 00:00:00                    FY2018   
4   MMM.N  2019  2019-12-31 00:00:00                    FY2019   

             Calc 

Section 4: Divide merged_with_ESG into "Suspected Multiple RICs from the Same Company / Multiple Names for the Same RIC" and "No Issues"

This section will perform your step 4.

Here, I first define "Problematic" as two types:

* Multiple different RICs with the same ETF + same year + same normalized company name

* Multiple different normalized company names with the same ETF + same year + same RIC

This will detect:

* Different listing locations/different share classes for the same company

* Changes in the same RIC name such as ORD/PDF

In [ ]:
# =========================
# Section 4: Splitting problematic / non_problematic
# =========================

spy_with_esg = read_table(os.path.join(MERGED_DIR, SPY_MERGED_ESG))
eem_with_esg = read_table(os.path.join(MERGED_DIR, EEM_MERGED_ESG))


def split_problematic_cases(df, etf_label):
    data = df.copy()

    # Create a normalized company name
    data["Name_norm"] = data["Name"].apply(normalize_text)

    # --------
    # Rule A: Multiple RICs with the same company name (after normalization) in the same year
    # --------
    a = (
        data.groupby(["etf", "year", "Name_norm"])["RIC"]
        .nunique(dropna=True)
        .reset_index(name="n_ric_per_name")
    )
    a["flag_name_multi_ric"] = a["n_ric_per_name"] > 1

    data = data.merge(
        a[["etf", "year", "Name_norm", "flag_name_multi_ric"]],
        on=["etf", "year", "Name_norm"],
        how="left"
    )

    # --------
    # Rule B: Same year, same RIC, multiple company names (after standardization)
    # --------
    b = (
        data.groupby(["etf", "year", "RIC"])["Name_norm"]
        .nunique(dropna=True)
        .reset_index(name="n_name_per_ric")
    )
    b["flag_ric_multi_name"] = b["n_name_per_ric"] > 1

    data = data.merge(
        b[["etf", "year", "RIC", "flag_ric_multi_name"]],
        on=["etf", "year", "RIC"],
        how="left"
    )

    data["flag_name_multi_ric"] = data["flag_name_multi_ric"].fillna(False)
    data["flag_ric_multi_name"] = data["flag_ric_multi_name"].fillna(False)

    data["problem_flag"] = data["flag_name_multi_ric"] | data["flag_ric_multi_name"]

    problematic = data[data["problem_flag"]].copy()
    clean = data[~data["problem_flag"]].copy()

    print(f"\n{etf_label} total rows: {len(data):,}")
    print(f"{etf_label} problematic rows: {len(problematic):,}")
    print(f"{etf_label} clean rows: {len(clean):,}")

    return data, problematic, clean


spy_all, spy_problematic, spy_clean = split_problematic_cases(spy_with_esg, "SPY")
eem_all, eem_problematic, eem_clean = split_problematic_cases(eem_with_esg, "EEM")

print("\nSPY problematic preview:")
print(
    spy_problematic[
        ["etf", "year", "RIC", "Name", "Name_norm", "flag_name_multi_ric", "flag_ric_multi_name"]
    ].head(20)
)

print("\nEEM problematic preview:")
print(
    eem_problematic[
        ["etf", "year", "RIC", "Name", "Name_norm", "flag_name_multi_ric", "flag_ric_multi_name"]
    ].head(20)
)

# save results
save_xlsx(spy_problematic, os.path.join(CLEAN_DIR, "SPY_problematic_cases.xlsx"))
save_xlsx(spy_clean, os.path.join(CLEAN_DIR, "SPY_clean_cases.xlsx"))

save_xlsx(eem_problematic, os.path.join(CLEAN_DIR, "EEM_problematic_cases.xlsx"))
save_xlsx(eem_clean, os.path.join(CLEAN_DIR, "EEM_clean_cases.xlsx"))


SPY total rows: 4,038
SPY problematic rows: 52
SPY clean rows: 3,986

EEM total rows: 8,335
EEM problematic rows: 621
EEM clean rows: 7,714

SPY problematic preview:
      etf  year       RIC                      Name     Name_norm  \
184   SPY  2015  GOOGL.OQ  ALPHABET INC CLASS A ORD  ALPHABET INC   
185   SPY  2016  GOOGL.OQ  ALPHABET INC CLASS A ORD  ALPHABET INC   
186   SPY  2017  GOOGL.OQ  ALPHABET INC CLASS A ORD  ALPHABET INC   
187   SPY  2018  GOOGL.OQ  ALPHABET INC CLASS A ORD  ALPHABET INC   
188   SPY  2019  GOOGL.OQ  ALPHABET INC CLASS A ORD  ALPHABET INC   
189   SPY  2020  GOOGL.OQ  ALPHABET INC CLASS A ORD  ALPHABET INC   
190   SPY  2021  GOOGL.OQ  ALPHABET INC CLASS A ORD  ALPHABET INC   
191   SPY  2022  GOOGL.OQ  ALPHABET INC CLASS A ORD  ALPHABET INC   
192   SPY  2015   GOOG.OQ  ALPHABET INC CLASS C ORD  ALPHABET INC   
193   SPY  2016   GOOG.OQ  ALPHABET INC CLASS C ORD  ALPHABET INC   
194   SPY  2017   GOOG.OQ  ALPHABET INC CLASS C ORD  ALPHABET INC   
195  

/tmp/ipykernel_10738/3148113864.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data["flag_ric_multi_name"] = data["flag_ric_multi_name"].fillna(False)


Saved: /content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/clean_data/SPY_clean_cases.xlsx
Saved: /content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/clean_data/EEM_problematic_cases.xlsx
Saved: /content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/clean_data/EEM_clean_cases.xlsx


After inspection, it was found that the ESG data should be lagged one phase before merging. Therefore, after manually checking the two files, the ESG data was deleted and merged again before processing the problematic data.

Step 1: Process the ESG (create lag_year)

This step only does one thing: Convert the year in the ESG to lag_year = year + 1

In [ ]:
import pandas as pd

# read ESG data
spy_esg = pd.read_excel(f"{SPY_DIR}/LSEG_ESG_SPY_2015_2024_v2.xlsx")
eem_esg = pd.read_excel(f"{EEM_DIR}/LSEG_ESG_EEM_2015_2024_v2.xlsx")

# lag_year
spy_esg["lag_year"] = spy_esg["year"] + 1
eem_esg["lag_year"] = eem_esg["year"] + 1

# check
print(spy_esg[["RIC", "year", "lag_year"]].head())
print(eem_esg[["RIC", "year", "lag_year"]].head())

     RIC    year  lag_year
0  MMM.N  2014.0    2015.0
1  MMM.N  2015.0    2016.0
2  MMM.N  2016.0    2017.0
3  MMM.N  2017.0    2018.0
4  MMM.N  2018.0    2019.0
         RIC    year  lag_year
0  601360.SS  2017.0    2018.0
1  601360.SS  2018.0    2019.0
2  601360.SS  2019.0    2020.0
3  601360.SS  2020.0    2021.0
4  601360.SS  2021.0    2022.0


Step 2: Merge the ESG into four datasets (using lag)

Merge key:

* Left: RIC + year
* Right: RIC + lag_year

In [ ]:
def merge_with_lag_esg(df, esg_df):
    merged = df.merge(
        esg_df,
        left_on=["RIC", "year"],
        right_on=["RIC", "lag_year"],
        how="left",
        suffixes=("", "_esg")
    )
    return merged

In [ ]:
# read files

spy_problem = pd.read_excel(f"{CLEAN_DIR}/SPY_problematic_cases.xlsx")
eem_problem = pd.read_excel(f"{CLEAN_DIR}/EEM_problematic_cases.xlsx")

spy_clean = pd.read_excel(f"{CLEAN_DIR}/SPY_clean_cases.xlsx")
eem_clean = pd.read_excel(f"{CLEAN_DIR}/EEM_clean_cases.xlsx")

In [ ]:
# merge ESG data

spy_problem = merge_with_lag_esg(spy_problem, spy_esg)
eem_problem = merge_with_lag_esg(eem_problem, eem_esg)

spy_clean = merge_with_lag_esg(spy_clean, spy_esg)
eem_clean = merge_with_lag_esg(eem_clean, eem_esg)

In [ ]:
spy_problem.to_excel(f"{CLEAN_DIR}/SPY_problem_merged_with_lag_esg.xlsx", index=False)
eem_problem.to_excel(f"{CLEAN_DIR}/EEM_problem_merged_with_lag_esg.xlsx", index=False)
spy_clean.to_excel(f"{CLEAN_DIR}/SPY_clean_merged_with_lag_esg.xlsx", index=False)
eem_clean.to_excel(f"{CLEAN_DIR}/EEM_clean_merged_with_lag_esg.xlsx", index=False)

In [ ]:
spy_problem.head()

,etf,date,RIC,Name,Country,Weight,No. Shares,old_RIC,year,Name_norm,...,Date,Financial Period Absolute,Calc Date,ESG Score,Environmental Pillar Score,Social Pillar Score,Governance Pillar Score,ESG Combined Score Grade,year_esg,lag_year
0,SPY,2015-12-31,GOOGL.OQ,ALPHABET INC CLASS A ORD,UNITED STATES,0.012625,2953962,GOOGL.OQ,2015,ALPHABET INC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SPY,2016-12-31,GOOGL.OQ,ALPHABET INC CLASS A ORD,UNITED STATES,0.012197,3462976,GOOGL.OQ,2016,ALPHABET INC,...,2015-12-31 00:00:00,FY2015,2015-12-31 00:00:00,62.159946,74.899425,51.109903,67.853416,C-,2015.0,2016.0
2,SPY,2017-12-31,GOOGL.OQ,ALPHABET INC CLASS A ORD,UNITED STATES,0.013443,3541822,GOOGL.OQ,2017,ALPHABET INC,...,2016-12-31 00:00:00,FY2016,2016-12-31 00:00:00,62.273915,76.465529,61.549945,58.639849,C-,2016.0,2017.0
3,SPY,2018-12-31,GOOGL.OQ,ALPHABET INC CLASS A ORD,UNITED STATES,0.014644,3410833,GOOGL.OQ,2018,ALPHABET INC,...,2017-12-31 00:00:00,FY2017,2017-12-31 00:00:00,59.252864,73.684512,71.851497,44.074547,C-,2017.0,2018.0
4,SPY,2019-12-31,GOOGL.OQ,ALPHABET INC CLASS A ORD,UNITED STATES,0.015224,3493875,GOOGL.OQ,2019,ALPHABET INC,...,2018-12-31 00:00:00,FY2018,2018-12-31 00:00:00,70.136369,73.946439,88.319141,53.335961,C,2018.0,2019.0


Step 3: Define the ESG field (which will be used later to determine whether the data can be merged).

In [ ]:
ESG_COLS = [
    "ESG Score",
    "Environmental Pillar Score",
    "Social Pillar Score",
    "Governance Pillar Score",
    "ESG Combined Score Grade"
]

Step 4: Find data with "ESG inconsistencies → cannot be merged"

Core logic:

If ESG values ​​are different within the same group → do not merge → save

In [ ]:
def find_esg_conflict(df):
    # Check if each group's ESG is unique
    conflict = df.groupby(["etf", "year", "normalize_name"])[ESG_COLS] \
        .nunique() \
        .reset_index()

    # Any column > 1 indicates a conflict
    for col in ESG_COLS:
        conflict = conflict[conflict[col] <= 1]

    return conflict

Step 5 (Important): Redefining the "Three Types of Problem Groups"

We need to first create a group ID.

In [ ]:
GROUP_COLS_1 = ["etf", "year", "Name_norm"]
GROUP_COLS_2 = ["etf", "year", "RIC"]
GROUP_COLS_3 = ["etf", "year", "RIC", "Name_norm"]

Step 6: Check if "each ESG group is consistent"

In [ ]:
def check_esg_consistency(df, group_cols):
    check = df.groupby(group_cols)[ESG_COLS] \
        .nunique() \
        .reset_index()

    # check wheather ESG are the consistent
    check["esg_consistent"] = (check[ESG_COLS] <= 1).all(axis=1)

    return check

In [ ]:
check1 = check_esg_consistency(spy_problem, GROUP_COLS_1)
check2 = check_esg_consistency(spy_problem, GROUP_COLS_2)
check3 = check_esg_consistency(spy_problem, GROUP_COLS_3)

In [ ]:
conflict1 = check1[check1["esg_consistent"] == False]
conflict2 = check2[check2["esg_consistent"] == False]
conflict3 = check3[check3["esg_consistent"] == False]

print(len(conflict1), len(conflict2), len(conflict3))

0 0 0


Step 7: Extract the "data that cannot be merged".

In [ ]:
def extract_conflict_rows(df, conflict_df, group_cols):
    merged = df.merge(conflict_df[group_cols], on=group_cols, how="inner")
    return merged

In [ ]:
spy_conflict_rows = pd.concat([
    extract_conflict_rows(spy_problem, conflict1, GROUP_COLS_1),
    extract_conflict_rows(spy_problem, conflict2, GROUP_COLS_2),
    extract_conflict_rows(spy_problem, conflict3, GROUP_COLS_3)
]).drop_duplicates()

In [ ]:
spy_conflict_rows.to_excel(f"{CLEAN_DIR}/SPY_esg_conflict_cases.xlsx", index=False)

In [ ]:
spy_problem.head()

,etf,date,RIC,Name,Country,Weight,No. Shares,old_RIC,year,Name_norm,...,Date,Financial Period Absolute,Calc Date,ESG Score,Environmental Pillar Score,Social Pillar Score,Governance Pillar Score,ESG Combined Score Grade,year_esg,lag_year
0,SPY,2015-12-31,GOOGL.OQ,ALPHABET INC CLASS A ORD,UNITED STATES,0.012625,2953962,GOOGL.OQ,2015,ALPHABET INC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SPY,2016-12-31,GOOGL.OQ,ALPHABET INC CLASS A ORD,UNITED STATES,0.012197,3462976,GOOGL.OQ,2016,ALPHABET INC,...,2015-12-31 00:00:00,FY2015,2015-12-31 00:00:00,62.159946,74.899425,51.109903,67.853416,C-,2015.0,2016.0
2,SPY,2017-12-31,GOOGL.OQ,ALPHABET INC CLASS A ORD,UNITED STATES,0.013443,3541822,GOOGL.OQ,2017,ALPHABET INC,...,2016-12-31 00:00:00,FY2016,2016-12-31 00:00:00,62.273915,76.465529,61.549945,58.639849,C-,2016.0,2017.0
3,SPY,2018-12-31,GOOGL.OQ,ALPHABET INC CLASS A ORD,UNITED STATES,0.014644,3410833,GOOGL.OQ,2018,ALPHABET INC,...,2017-12-31 00:00:00,FY2017,2017-12-31 00:00:00,59.252864,73.684512,71.851497,44.074547,C-,2017.0,2018.0
4,SPY,2019-12-31,GOOGL.OQ,ALPHABET INC CLASS A ORD,UNITED STATES,0.015224,3493875,GOOGL.OQ,2019,ALPHABET INC,...,2018-12-31 00:00:00,FY2018,2018-12-31 00:00:00,70.136369,73.946439,88.319141,53.335961,C,2018.0,2019.0


In [ ]:
spy_conflict_rows.head()

,etf,date,RIC,Name,Country,Weight,No. Shares,old_RIC,year,Name_norm,...,Date,Financial Period Absolute,Calc Date,ESG Score,Environmental Pillar Score,Social Pillar Score,Governance Pillar Score,ESG Combined Score Grade,year_esg,lag_year


Step 8: Remaining "Data that can be merged"

In [ ]:
spy_valid = spy_problem.drop(spy_conflict_rows.index)

In [ ]:
spy_valid

,etf,date,RIC,Name,Country,Weight,No. Shares,old_RIC,year,Name_norm,...,Date,Financial Period Absolute,Calc Date,ESG Score,Environmental Pillar Score,Social Pillar Score,Governance Pillar Score,ESG Combined Score Grade,year_esg,lag_year
0,SPY,2015-12-31,GOOGL.OQ,ALPHABET INC CLASS A ORD,UNITED STATES,0.012625,2953962,GOOGL.OQ,2015,ALPHABET INC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SPY,2016-12-31,GOOGL.OQ,ALPHABET INC CLASS A ORD,UNITED STATES,0.012197,3462976,GOOGL.OQ,2016,ALPHABET INC,...,2015-12-31 00:00:00,FY2015,2015-12-31 00:00:00,62.159946,74.899425,51.109903,67.853416,C-,2015.0,2016.0
2,SPY,2017-12-31,GOOGL.OQ,ALPHABET INC CLASS A ORD,UNITED STATES,0.013443,3541822,GOOGL.OQ,2017,ALPHABET INC,...,2016-12-31 00:00:00,FY2016,2016-12-31 00:00:00,62.273915,76.465529,61.549945,58.639849,C-,2016.0,2017.0
3,SPY,2018-12-31,GOOGL.OQ,ALPHABET INC CLASS A ORD,UNITED STATES,0.014644,3410833,GOOGL.OQ,2018,ALPHABET INC,...,2017-12-31 00:00:00,FY2017,2017-12-31 00:00:00,59.252864,73.684512,71.851497,44.074547,C-,2017.0,2018.0
4,SPY,2019-12-31,GOOGL.OQ,ALPHABET INC CLASS A ORD,UNITED STATES,0.015224,3493875,GOOGL.OQ,2019,ALPHABET INC,...,2018-12-31 00:00:00,FY2018,2018-12-31 00:00:00,70.136369,73.946439,88.319141,53.335961,C,2018.0,2019.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,SPY,2019-12-31,DISCK.O^D22,WARNER BROS DISCOVERY INC ORD,UNITED STATES,0.000195,1831133,DISCK.O^D22,2019,WARNER BROS DISCOVERY INC,...,2018-12-31 00:00:00,FY2018,2018-12-31 00:00:00,42.309553,9.244186,59.675682,30.465938,C+,2018.0,2019.0
62,SPY,2020-12-31,DISCK.O^D22,WARNER BROS DISCOVERY INC ORD,UNITED STATES,0.000247,3100218,DISCK.O^D22,2020,WARNER BROS DISCOVERY INC,...,2019-12-31 00:00:00,FY2019,2019-12-31 00:00:00,49.644976,8.112245,59.216125,50.887207,C+,2019.0,2020.0
63,SPY,2020-12-31,DISCK.O^D22,WARNER BROS DISCOVERY INC ORD,UNITED STATES,0.000154,1683653,DISCK.O^D22,2020,WARNER BROS DISCOVERY INC,...,2019-12-31 00:00:00,FY2019,2019-12-31 00:00:00,49.644976,8.112245,59.216125,50.887207,C+,2019.0,2020.0
64,SPY,2021-12-31,DISCK.O^D22,WARNER BROS DISCOVERY INC ORD,UNITED STATES,0.000166,3301146,DISCK.O^D22,2021,WARNER BROS DISCOVERY INC,...,2020-12-31 00:00:00,FY2020,2020-12-31 00:00:00,50.652623,11.440678,56.072689,56.586850,B-,2020.0,2021.0


Paragraph 9: First, label the three types

This paragraph will label the SPY problematic data as type1 / type2 / type3, allowing you to see how many entries there are for each type.

In [ ]:
import pandas as pd

# Reread to avoid being affected by the previous notebook status.
spy_problem = pd.read_excel(f"{CLEAN_DIR}/SPY_problematic_cases.xlsx")
spy_esg = pd.read_excel(f"{SPY_DIR}/LSEG_ESG_SPY_2015_2024_v2.xlsx")

# ESG lag
spy_esg["lag_year"] = spy_esg["year"] + 1

# merge ESG
spy_problem = spy_problem.merge(
    spy_esg,
    left_on=["RIC", "year"],
    right_on=["RIC", "lag_year"],
    how="left",
    suffixes=("", "_esg")
)

ESG_COLS = [
    "ESG Score",
    "Environmental Pillar Score",
    "Social Pillar Score",
    "Governance Pillar Score",
    "ESG Combined Score Grade"
]

# definite three problems
g1 = spy_problem.groupby(["etf", "year", "Name_norm"])["RIC"].nunique().reset_index(name="n_ric")
g1["type1_flag"] = g1["n_ric"] > 1

g2 = spy_problem.groupby(["etf", "year", "RIC"])["Name_norm"].nunique().reset_index(name="n_name")
g2["type2_flag"] = g2["n_name"] > 1

g3 = spy_problem.groupby(["etf", "year", "RIC", "Name_norm"]).size().reset_index(name="n_rows")
g3["type3_flag"] = g3["n_rows"] > 1

spy_problem = spy_problem.merge(
    g1[["etf", "year", "Name_norm", "type1_flag"]],
    on=["etf", "year", "Name_norm"],
    how="left"
)

spy_problem = spy_problem.merge(
    g2[["etf", "year", "RIC", "type2_flag"]],
    on=["etf", "year", "RIC"],
    how="left"
)

spy_problem = spy_problem.merge(
    g3[["etf", "year", "RIC", "Name_norm", "type3_flag"]],
    on=["etf", "year", "RIC", "Name_norm"],
    how="left"
)

spy_problem["type1_flag"] = spy_problem["type1_flag"].fillna(False)
spy_problem["type2_flag"] = spy_problem["type2_flag"].fillna(False)
spy_problem["type3_flag"] = spy_problem["type3_flag"].fillna(False)

print("type1 rows:", spy_problem["type1_flag"].sum())
print("type2 rows:", spy_problem["type2_flag"].sum())
print("type3 rows:", spy_problem["type3_flag"].sum())

spy_problem[[
    "etf", "year", "RIC", "Name", "Name_norm",
    "type1_flag", "type2_flag", "type3_flag"
]].head(20)

type1 rows: 52
type2 rows: 0
type3 rows: 14


,etf,year,RIC,Name,Name_norm,type1_flag,type2_flag,type3_flag
0,SPY,2015,GOOGL.OQ,ALPHABET INC CLASS A ORD,ALPHABET INC,True,False,False
1,SPY,2016,GOOGL.OQ,ALPHABET INC CLASS A ORD,ALPHABET INC,True,False,False
2,SPY,2017,GOOGL.OQ,ALPHABET INC CLASS A ORD,ALPHABET INC,True,False,False
3,SPY,2018,GOOGL.OQ,ALPHABET INC CLASS A ORD,ALPHABET INC,True,False,False
4,SPY,2019,GOOGL.OQ,ALPHABET INC CLASS A ORD,ALPHABET INC,True,False,False
5,SPY,2020,GOOGL.OQ,ALPHABET INC CLASS A ORD,ALPHABET INC,True,False,False
6,SPY,2021,GOOGL.OQ,ALPHABET INC CLASS A ORD,ALPHABET INC,True,False,False
7,SPY,2022,GOOGL.OQ,ALPHABET INC CLASS A ORD,ALPHABET INC,True,False,False
8,SPY,2015,GOOG.OQ,ALPHABET INC CLASS C ORD,ALPHABET INC,True,False,False
9,SPY,2016,GOOG.OQ,ALPHABET INC CLASS C ORD,ALPHABET INC,True,False,False


Section 10: Check if the ESG of each group is consistent.

This section will identify groups with inconsistent ESGs within the same group.

If it's still empty, it means these problematic cases can be merged.

In [ ]:
def esg_consistency_check(df, group_cols, esg_cols):
    check = df.groupby(group_cols)[esg_cols].nunique(dropna=False).reset_index()
    check["esg_consistent"] = (check[esg_cols] <= 1).all(axis=1)
    return check

# type1: Same as Name_norm for multiple RICs
check_type1 = esg_consistency_check(
    spy_problem[spy_problem["type1_flag"]],
    ["etf", "year", "Name_norm"],
    ESG_COLS
)

# type2: Same as RIC multiple Name_norm
check_type2 = esg_consistency_check(
    spy_problem[spy_problem["type2_flag"]],
    ["etf", "year", "RIC"],
    ESG_COLS
)

# type3: Same as RIC + Same as Name_norm with duplicate columns
check_type3 = esg_consistency_check(
    spy_problem[spy_problem["type3_flag"]],
    ["etf", "year", "RIC", "Name_norm"],
    ESG_COLS
)

type1_conflict = check_type1[~check_type1["esg_consistent"]]
type2_conflict = check_type2[~check_type2["esg_consistent"]]
type3_conflict = check_type3[~check_type3["esg_consistent"]]

print("type1 ESG conflicts:", len(type1_conflict))
print("type2 ESG conflicts:", len(type2_conflict))
print("type3 ESG conflicts:", len(type3_conflict))

print("\ntype1 conflict preview:")
print(type1_conflict.head())

print("\ntype2 conflict preview:")
print(type2_conflict.head())

print("\ntype3 conflict preview:")
print(type3_conflict.head())

type1 ESG conflicts: 0
type2 ESG conflicts: 0
type3 ESG conflicts: 0

type1 conflict preview:
Empty DataFrame
Columns: [etf, year, Name_norm, ESG Score, Environmental Pillar Score, Social Pillar Score, Governance Pillar Score, ESG Combined Score Grade, esg_consistent]
Index: []

type2 conflict preview:
Empty DataFrame
Columns: [etf, year, RIC, ESG Score, Environmental Pillar Score, Social Pillar Score, Governance Pillar Score, ESG Combined Score Grade, esg_consistent]
Index: []

type3 conflict preview:
Empty DataFrame
Columns: [etf, year, RIC, Name_norm, ESG Score, Environmental Pillar Score, Social Pillar Score, Governance Pillar Score, ESG Combined Score Grade, esg_consistent]
Index: []


Paragraph 12: Store the items that cannot be merged separately first.

In [ ]:
def extract_by_group(df, group_df, group_cols):
    if len(group_df) == 0:
        return df.iloc[0:0].copy()
    return df.merge(group_df[group_cols], on=group_cols, how="inner")

spy_type1_conflict_rows = extract_by_group(
    spy_problem, type1_conflict, ["etf", "year", "Name_norm"]
)

spy_type2_conflict_rows = extract_by_group(
    spy_problem, type2_conflict, ["etf", "year", "RIC"]
)

spy_type3_conflict_rows = extract_by_group(
    spy_problem, type3_conflict, ["etf", "year", "RIC", "Name_norm"]
)

spy_all_conflict_rows = pd.concat([
    spy_type1_conflict_rows,
    spy_type2_conflict_rows,
    spy_type3_conflict_rows
]).drop_duplicates()

print("all SPY conflict rows:", len(spy_all_conflict_rows))
print(spy_all_conflict_rows.head())

spy_all_conflict_rows.to_excel(
    f"{CLEAN_DIR}/SPY_problematic_esg_conflict_cases.xlsx",
    index=False
)

all SPY conflict rows: 0
Empty DataFrame
Columns: [etf, date, RIC, Name, Country, Weight, No. Shares, old_RIC, year, Name_norm, flag_name_multi_ric, flag_ric_multi_name, problem_flag, Date, Financial Period Absolute, Calc Date, ESG Score, Environmental Pillar Score, Social Pillar Score, Governance Pillar Score, ESG Combined Score Grade, year_esg, lag_year, type1_flag, type2_flag, type3_flag]
Index: []

[0 rows x 26 columns]


Paragraph 13: Retain the SPY problematic data that can be merged.

In [ ]:
if len(spy_all_conflict_rows) == 0:
    spy_problem_ok = spy_problem.copy()
else:
    spy_problem_ok = spy_problem.merge(
        spy_all_conflict_rows[["etf", "date", "RIC", "Name", "year"]],
        on=["etf", "date", "RIC", "Name", "year"],
        how="left",
        indicator=True
    )
    spy_problem_ok = spy_problem_ok[spy_problem_ok["_merge"] == "left_only"].drop(columns=["_merge"])

print("SPY problem rows that can be merged:", len(spy_problem_ok))
print(spy_problem_ok[[
    "etf", "year", "RIC", "Name", "Name_norm",
    "type1_flag", "type2_flag", "type3_flag"
]].head(20))

SPY problem rows that can be merged: 66
    etf  year       RIC                      Name     Name_norm  type1_flag  \
0   SPY  2015  GOOGL.OQ  ALPHABET INC CLASS A ORD  ALPHABET INC        True   
1   SPY  2016  GOOGL.OQ  ALPHABET INC CLASS A ORD  ALPHABET INC        True   
2   SPY  2017  GOOGL.OQ  ALPHABET INC CLASS A ORD  ALPHABET INC        True   
3   SPY  2018  GOOGL.OQ  ALPHABET INC CLASS A ORD  ALPHABET INC        True   
4   SPY  2019  GOOGL.OQ  ALPHABET INC CLASS A ORD  ALPHABET INC        True   
5   SPY  2020  GOOGL.OQ  ALPHABET INC CLASS A ORD  ALPHABET INC        True   
6   SPY  2021  GOOGL.OQ  ALPHABET INC CLASS A ORD  ALPHABET INC        True   
7   SPY  2022  GOOGL.OQ  ALPHABET INC CLASS A ORD  ALPHABET INC        True   
8   SPY  2015   GOOG.OQ  ALPHABET INC CLASS C ORD  ALPHABET INC        True   
9   SPY  2016   GOOG.OQ  ALPHABET INC CLASS C ORD  ALPHABET INC        True   
10  SPY  2017   GOOG.OQ  ALPHABET INC CLASS C ORD  ALPHABET INC        True   
11  SPY  201

Paragraph 14: Truly Merging SPY Problematic Cases

I'll apply the rules for this step based on your current definition:

* Type 3: Merge directly into one transaction

* Type 1 / Type 2: Merge only if ESG is consistent

Weights and number of shares are summed

RIC and Name are initially retained based on the first transaction in each group

If there are multiple RICs in the same group, connect them with | for easier checking later.

In [ ]:
def concat_unique(series):
    vals = pd.Series(series).dropna().astype(str).unique().tolist()
    return " | ".join(vals)

# Define which columns to sum
sum_cols = []
for c in ["Weight", "No. Shares"]:
    if c in spy_problem_ok.columns:
        sum_cols.append(c)

# Keep the first / for other columns
agg_dict = {
    "date": "first",
    "RIC": "first",
    "Name": "first",
    "Country": "first",
    "old_RIC": concat_unique,
    "Name_norm": "first",
    "type1_flag": "max",
    "type2_flag": "max",
    "type3_flag": "max",
}

for c in sum_cols:
    agg_dict[c] = "sum"

# Keep the first entry in the ESG column
for c in ESG_COLS:
    if c in spy_problem_ok.columns:
        agg_dict[c] = "first"

for c in ["Financial Period Absolute", "Date", "Calc Date", "year_esg", "lag_year"]:
    if c in spy_problem_ok.columns:
        agg_dict[c] = "first"

# Merge using company hierarchy
# There are issues because they all belong to the same company, so we'll use etf + year + Name_norm here first.
spy_problem_merged = (
    spy_problem_ok
    .groupby(["etf", "year", "Name_norm"], as_index=False)
    .agg(agg_dict)
)

print("SPY problematic original rows:", len(spy_problem_ok))
print("SPY problematic merged rows:", len(spy_problem_merged))

print(spy_problem_merged.head(20))

SPY problematic original rows: 66
SPY problematic merged rows: 33
    etf  year        date          RIC                           Name  \
0   SPY  2015  2015-12-31     GOOGL.OQ       ALPHABET INC CLASS A ORD   
1   SPY  2015  2015-12-31      NWSA.OQ                  NEWS CORP ORD   
2   SPY  2015  2015-12-31  DISCK.O^D22  WARNER BROS DISCOVERY INC ORD   
3   SPY  2016  2016-12-31     GOOGL.OQ       ALPHABET INC CLASS A ORD   
4   SPY  2016  2016-12-31      NWSA.OQ                  NEWS CORP ORD   
5   SPY  2016  2016-12-31        UAA.N           UNDER ARMOUR INC ORD   
6   SPY  2016  2016-12-31  DISCK.O^D22  WARNER BROS DISCOVERY INC ORD   
7   SPY  2017  2017-12-31     GOOGL.OQ       ALPHABET INC CLASS A ORD   
8   SPY  2017  2017-12-31      NWSA.OQ                  NEWS CORP ORD   
9   SPY  2017  2017-12-31        UAA.N           UNDER ARMOUR INC ORD   
10  SPY  2017  2017-12-31  DISCK.O^D22  WARNER BROS DISCOVERY INC ORD   
11  SPY  2018  2018-12-31     GOOGL.OQ       ALPHABET INC 

Paragraph 15: Combine the problematic entries from the SPY clean + merge files and save them as a new file.

In [ ]:
spy_clean = pd.read_excel(f"{CLEAN_DIR}/SPY_clean_cases.xlsx")

# Clean also adds lag ESG to ensure consistent field alignment.
spy_clean = spy_clean.merge(
    spy_esg,
    left_on=["RIC", "year"],
    right_on=["RIC", "lag_year"],
    how="left",
    suffixes=("", "_esg")
)

spy_final = pd.concat([spy_clean, spy_problem_merged], ignore_index=True, sort=False)

print("SPY final shape:", spy_final.shape)
print(spy_final.head())

spy_final.to_excel(f"{CLEAN_DIR}/SPY_final_cleaned_with_lag_esg.xlsx", index=False)

SPY final shape: (4005, 26)
   etf        date    RIC       Name        Country    Weight  No. Shares  \
0  SPY  2015-12-31  MMM.N  3M CO ORD  UNITED STATES  0.005164     6240677   
1  SPY  2016-12-31  MMM.N  3M CO ORD  UNITED STATES  0.005586     7038215   
2  SPY  2017-12-31  MMM.N  3M CO ORD  UNITED STATES  0.006003     7078439   
3  SPY  2018-12-31  MMM.N  3M CO ORD  UNITED STATES  0.005205     6649048   
4  SPY  2019-12-31  MMM.N  3M CO ORD  UNITED STATES  0.003840     6691310   

  old_RIC  year Name_norm  ...  ESG Score Environmental Pillar Score  \
0   MMM.N  2015     3M CO  ...  89.996881                  84.196906   
1   MMM.N  2016     3M CO  ...  86.514077                  83.376100   
2   MMM.N  2017     3M CO  ...  88.128389                  85.960218   
3   MMM.N  2018     3M CO  ...  88.869561                  90.306540   
4   MMM.N  2019     3M CO  ...  87.612642                  91.397917   

  Social Pillar Score Governance Pillar Score ESG Combined Score Grade  \
0 

Step 16: Directly copy the SPY process to EEM

In [ ]:
# 1. read data + merge lag ESG

eem_problem = pd.read_excel(f"{CLEAN_DIR}/EEM_problematic_cases.xlsx")
eem_clean = pd.read_excel(f"{CLEAN_DIR}/EEM_clean_cases.xlsx")
eem_esg = pd.read_excel(f"{EEM_DIR}/LSEG_ESG_EEM_2015_2024_v2.xlsx")

eem_esg["lag_year"] = eem_esg["year"] + 1

def merge_with_lag_esg(df, esg):
    return df.merge(
        esg,
        left_on=["RIC", "year"],
        right_on=["RIC", "lag_year"],
        how="left",
        suffixes=("", "_esg")
    )

eem_problem = merge_with_lag_esg(eem_problem, eem_esg)
eem_clean = merge_with_lag_esg(eem_clean, eem_esg)

print("EEM problem shape:", eem_problem.shape)
print("EEM clean shape:", eem_clean.shape)

EEM problem shape: (648, 23)
EEM clean shape: (7686, 23)


In [ ]:
# 2. tag three types of problem

ESG_COLS = [
    "ESG Score",
    "Environmental Pillar Score",
    "Social Pillar Score",
    "Governance Pillar Score",
    "ESG Combined Score Grade"
]

g1 = eem_problem.groupby(["etf", "year", "Name_norm"])["RIC"].nunique().reset_index(name="n_ric")
g1["type1_flag"] = g1["n_ric"] > 1

g2 = eem_problem.groupby(["etf", "year", "RIC"])["Name_norm"].nunique().reset_index(name="n_name")
g2["type2_flag"] = g2["n_name"] > 1

g3 = eem_problem.groupby(["etf", "year", "RIC", "Name_norm"]).size().reset_index(name="n_rows")
g3["type3_flag"] = g3["n_rows"] > 1

eem_problem = eem_problem.merge(g1[["etf","year","Name_norm","type1_flag"]], on=["etf","year","Name_norm"], how="left")
eem_problem = eem_problem.merge(g2[["etf","year","RIC","type2_flag"]], on=["etf","year","RIC"], how="left")
eem_problem = eem_problem.merge(g3[["etf","year","RIC","Name_norm","type3_flag"]], on=["etf","year","RIC","Name_norm"], how="left")

eem_problem = eem_problem.fillna({"type1_flag":False,"type2_flag":False,"type3_flag":False})

print("type1:", eem_problem["type1_flag"].sum())
print("type2:", eem_problem["type2_flag"].sum())
print("type3:", eem_problem["type3_flag"].sum())

type1: 615
type2: 6
type3: 21


In [ ]:
# Classified data
eem_type1 = eem_problem[eem_problem["type1_flag"]].copy()
eem_type2 = eem_problem[eem_problem["type2_flag"]].copy()
eem_type3 = eem_problem[eem_problem["type3_flag"]].copy()

# Basic check
print("type1 rows:", len(eem_type1))
print("type2 rows:", len(eem_type2))
print("type3 rows:", len(eem_type3))

eem_type1.head()

type1 rows: 615
type2 rows: 6
type3 rows: 21


,etf,date,RIC,Name,Country,Weight,No. Shares,old_RIC,year,Name_norm,...,ESG Score,Environmental Pillar Score,Social Pillar Score,Governance Pillar Score,ESG Combined Score Grade,year_esg,lag_year,type1_flag,type2_flag,type3_flag
0,EEM,2018-12-31,1288.HK,AGRICULTURAL BANK OF CHINA LTD ORD,CHINA,0.002578,172490000,1288.HK,2018,AGRICULTURAL BANK OF CHINA LTD,...,55.446206,72.500364,38.901137,79.458598,B-,2017.0,2018.0,True,False,False
1,EEM,2018-12-31,601288.SS,AGRICULTURAL BANK OF CHINA LTD ORD,CHINA,0.000219,12233500,601288.SS,2018,AGRICULTURAL BANK OF CHINA LTD,...,55.446206,72.500364,38.901137,79.458598,B-,2017.0,2018.0,True,False,False
2,EEM,2019-12-31,1288.HK,AGRICULTURAL BANK OF CHINA LTD ORD,CHINA,0.001950,132775000,1288.HK,2019,AGRICULTURAL BANK OF CHINA LTD,...,47.911880,71.074044,32.729306,67.920136,C+,2018.0,2019.0,True,False,False
3,EEM,2019-12-31,601288.SS,AGRICULTURAL BANK OF CHINA LTD ORD,CHINA,0.000396,22396300,601288.SS,2019,AGRICULTURAL BANK OF CHINA LTD,...,47.911880,71.074044,32.729306,67.920136,C+,2018.0,2019.0,True,False,False
4,EEM,2020-12-31,1288.HK,AGRICULTURAL BANK OF CHINA LTD ORD,CHINA,0.001270,98157000,1288.HK,2020,AGRICULTURAL BANK OF CHINA LTD,...,52.627393,74.058781,33.177392,78.657114,B-,2019.0,2020.0,True,False,False


In [ ]:
eem_type1.to_excel(f"{CLEAN_DIR}/EEM_type1_same_name_multi_RIC.xlsx", index=False)
eem_type2.to_excel(f"{CLEAN_DIR}/EEM_type2_same_RIC_multi_name.xlsx", index=False)
eem_type3.to_excel(f"{CLEAN_DIR}/EEM_type3_exact_duplicate.xlsx", index=False)

print("type1 / type2 / type3 saved")

type1 / type2 / type3 saved


In [ ]:
def esg_check(df, group_cols):
    check = df.groupby(group_cols)[ESG_COLS].nunique().reset_index()
    check["consistent"] = (check[ESG_COLS] <= 1).all(axis=1)
    return check

c1 = esg_check(eem_problem[eem_problem["type1_flag"]], ["etf","year","Name_norm"])
c2 = esg_check(eem_problem[eem_problem["type2_flag"]], ["etf","year","RIC"])
c3 = esg_check(eem_problem[eem_problem["type3_flag"]], ["etf","year","RIC","Name_norm"])

conflict1 = c1[~c1["consistent"]]
conflict2 = c2[~c2["consistent"]]
conflict3 = c3[~c3["consistent"]]

print("conflict1:", len(conflict1))
print("conflict2:", len(conflict2))
print("conflict3:", len(conflict3))

conflict1: 4
conflict2: 0
conflict3: 0


In [ ]:
conflict1

,etf,year,Name_norm,ESG Score,Environmental Pillar Score,Social Pillar Score,Governance Pillar Score,ESG Combined Score Grade,consistent
2,EEM,2015,CHINA LIFE INSURANCE CO LTD,2,2,2,2,2,False
7,EEM,2015,MIRAE ASSET SECURITIES CO LTD,2,2,2,2,2,False
14,EEM,2016,CHINA LIFE INSURANCE CO LTD,2,2,2,2,2,False
27,EEM,2017,CHINA LIFE INSURANCE CO LTD,2,2,2,2,2,False


In [ ]:
def extract(df, conflict_df, keys):
    if len(conflict_df)==0:
        return df.iloc[0:0]
    return df.merge(conflict_df[keys], on=keys, how="inner")

eem_conflict = pd.concat([
    extract(eem_problem, conflict1, ["etf","year","Name_norm"]),
    extract(eem_problem, conflict2, ["etf","year","RIC"]),
    extract(eem_problem, conflict3, ["etf","year","RIC","Name_norm"])
]).drop_duplicates()

print("EEM ESG conflict rows:", len(eem_conflict))

eem_conflict.to_excel(f"{CLEAN_DIR}/EEM_esg_conflict_cases.xlsx", index=False)

EEM ESG conflict rows: 8


In [ ]:
eem_ok = eem_problem.drop(eem_conflict.index)
print("EEM valid rows:", len(eem_ok))

EEM valid rows: 640


Step 17: Merge EEM problematic

In [ ]:
def concat_unique(series):
    return " | ".join(pd.Series(series).dropna().astype(str).unique())

agg_dict = {
    "date":"first",
    "RIC":"first",
    "Name":"first",
    "Country":"first",
    "old_RIC":concat_unique,
    "Name_norm":"first",
    "type1_flag":"max",
    "type2_flag":"max",
    "type3_flag":"max",
    "Weight":"sum",
    "No. Shares":"sum"
}

for c in ESG_COLS:
    if c in eem_ok.columns:
        agg_dict[c] = "first"

eem_merged = eem_ok.groupby(["etf","year","Name_norm"], as_index=False).agg(agg_dict)

print("before:", len(eem_ok))
print("after:", len(eem_merged))

eem_merged.head()

before: 640
after: 321


,etf,year,date,RIC,Name,Country,old_RIC,Name_norm,type1_flag,type2_flag,type3_flag,Weight,No. Shares,ESG Score,Environmental Pillar Score,Social Pillar Score,Governance Pillar Score,ESG Combined Score Grade
0,EEM,2015,2015-12-31,090435.KS,AMOREPACIFIC CORP,KOREA,090435.KS | 090430.KS,AMOREPACIFIC CORP,True,False,False,0.003796,252702,58.818083,44.554203,66.021954,59.074436,B
1,EEM,2015,2015-12-31,BBDC4.SA,BANCO BRADESCO SA,BRAZIL,BBDC4.SA | BBDC3.SA,BANCO BRADESCO SA,True,False,False,0.004675,20496488,72.708041,83.156907,72.415203,72.372464,B+
2,EEM,2015,2015-12-31,AXIA3.SA,CENTRAIS ELETRICAS BRASILEIRAS SA - ELETROBRAS,BRAZIL,AXIA3.SA,CENTRAIS ELETRICAS BRASILEIRAS SA ELETROBRAS,False,False,False,0.000184,1497337,NaN,NaN,NaN,NaN,None
3,EEM,2015,2015-12-31,2628.HK,CHINA LIFE INSURANCE CO LTD ORD,CHINA,2628.HK | TW0002823002.TRE^F21,CHINA LIFE INSURANCE CO LTD,True,False,False,0.007665,66102983,43.113337,27.262895,39.846886,58.523467,C+
4,EEM,2015,2015-12-31,EUR.WA,EUROBANK SA ORD,GREECE,EUR.WA,EUROBANK SA,False,True,False,0.000591,11269029,NaN,NaN,NaN,NaN,None


In [ ]:
eem_final = pd.concat([eem_clean, eem_merged], ignore_index=True)

print("EEM final shape:", eem_final.shape)

eem_final.to_excel(f"{CLEAN_DIR}/EEM_final_cleaned_with_lag_esg.xlsx", index=False)

EEM final shape: (8007, 26)


After manually reorganizing the EEM_esg_conflict_cases and EEM_final_cleaned_with_lag_esg data sets, we merged the two sets together and then combined them with the newly acquired data from 2014.

In [ ]:
import os
import pandas as pd
import re

# Paragraph 1: First organize the 2014 ESG files and create year and lag_year


# =========================
# path setting
# =========================
MERGED_DIR = "/content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/merged_holdings"
SPY_DIR = "/content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/etf spy data"
EEM_DIR = "/content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/etf eem data"
CLEAN_DIR = "/content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/clean_data"

os.makedirs(CLEAN_DIR, exist_ok=True)

# path
spy_2014_path = f"{SPY_DIR}/LSEG_ESG_SPY_2014.xlsx"
eem_2014_path = f"{EEM_DIR}/LSEG_ESG_EEM_2014.xlsx"

# read files
spy_esg_2014 = pd.read_excel(spy_2014_path)
eem_esg_2014 = pd.read_excel(eem_2014_path)

# remove blank
spy_esg_2014.columns = [str(c).strip() for c in spy_esg_2014.columns]
eem_esg_2014.columns = [str(c).strip() for c in eem_esg_2014.columns]

print("SPY 2014 ESG columns:")
print(spy_esg_2014.columns.tolist())

print("\nEEM 2014 ESG columns:")
print(eem_esg_2014.columns.tolist())

SPY 2014 ESG columns:
['RIC', 'Date', 'Financial Period Absolute', 'Calc Date', 'ESG Score', 'Environmental Pillar Score', 'Social Pillar Score', 'Governance Pillar Score', 'ESG Combined Score Grade', 'year']

EEM 2014 ESG columns:
['RIC', 'Date', 'Financial Period Absolute', 'Calc Date', 'ESG Score', 'Environmental Pillar Score', 'Social Pillar Score', 'Governance Pillar Score', 'ESG Combined Score Grade', 'year']


In [ ]:
def extract_year(value):
    if pd.isna(value):
        return pd.NA
    m = re.search(r"(20\d{2})", str(value))
    return int(m.group(1)) if m else pd.NA

def clean_2014_esg(esg_df):
    df = esg_df.copy()

    # convert "NULL" into NA
    df = df.replace("NULL", pd.NA)

    # year into number data
    if "year" in df.columns:
        df["year"] = pd.to_numeric(df["year"], errors="coerce")
    elif "Financial Period Absolute" in df.columns:
        df["year"] = df["Financial Period Absolute"].apply(extract_year)
    else:
        raise KeyError("couldn't find column name year or Financial Period Absolute")

    # lag_year
    df["lag_year"] = df["year"] + 1

    # RIC sortind
    if "RIC" not in df.columns:
        raise KeyError("couldn't find column name RIC")
    df["RIC"] = df["RIC"].astype(str).str.strip()
    df.loc[df["RIC"].str.lower() == "nan", "RIC"] = pd.NA
    df.loc[df["RIC"] == "", "RIC"] = pd.NA

    df = df.drop_duplicates(subset=["RIC", "lag_year"]).reset_index(drop=True)

    return df

spy_esg_2014_clean = clean_2014_esg(spy_esg_2014)
eem_esg_2014_clean = clean_2014_esg(eem_esg_2014)

print("\nSPY 2014 ESG preview:")
print(spy_esg_2014_clean[["RIC", "year", "lag_year"]].head())

print("\nEEM 2014 ESG preview:")
print(eem_esg_2014_clean[["RIC", "year", "lag_year"]].head())


SPY 2014 ESG preview:
         RIC    year  lag_year
0      MMM.N  2013.0    2014.0
1      ABT.N  2013.0    2014.0
2     ABBV.N  2013.0    2014.0
3      ACN.N  2014.0    2015.0
4  ATVIn.BCU  2014.0    2015.0

EEM 2014 ESG preview:
          RIC    year  lag_year
0  WUBA.K^I20     NaN       NaN
1     2018.HK  2013.0    2014.0
2      AEV.PS  2014.0    2015.0
3       AP.PS  2014.0    2015.0
4      ABGJ.J  2014.0    2015.0


In [ ]:
spy_esg_2014_clean.to_excel(f"{SPY_DIR}/LSEG_ESG_SPY_2014_v2.xlsx", index=False)
eem_esg_2014_clean.to_excel(f"{EEM_DIR}/LSEG_ESG_EEM_2014_v2.xlsx", index=False)

print("Saved 2014 ESG cleaned files.")

Saved 2014 ESG cleaned files.


Section 2: Add the 2014 ESG to the current final file

The logic for this step is:

Read in your current final file

Find the columns in the ESG that are still empty

Use RIC + year to calculate RIC + lag_year

Only fill in the empty ESG columns

Save as a new version

In [ ]:
# read final cleaned files
spy_final = pd.read_excel(f"{CLEAN_DIR}/SPY_final_cleaned_with_lag_esg.xlsx")
eem_final = pd.read_excel(f"{CLEAN_DIR}/EEM_final_cleaned_with_lag_esg_v2.xlsx")

print("SPY final shape:", spy_final.shape)
print("EEM final shape:", eem_final.shape)

print("\nSPY final columns:")
print(spy_final.columns.tolist())

print("\nEEM final columns:")
print(eem_final.columns.tolist())

SPY final shape: (4005, 26)
EEM final shape: (8017, 26)

SPY final columns:
['etf', 'date', 'RIC', 'Name', 'Country', 'Weight', 'No. Shares', 'old_RIC', 'year', 'Name_norm', 'flag_name_multi_ric', 'flag_ric_multi_name', 'problem_flag', 'Date', 'Financial Period Absolute', 'Calc Date', 'ESG Score', 'Environmental Pillar Score', 'Social Pillar Score', 'Governance Pillar Score', 'ESG Combined Score Grade', 'year_esg', 'lag_year', 'type1_flag', 'type2_flag', 'type3_flag']

EEM final columns:
['etf', 'date', 'RIC', 'Name', 'Country', 'Weight', 'No. Shares', 'old_RIC', 'year', 'Name_norm', 'flag_name_multi_ric', 'flag_ric_multi_name', 'problem_flag', 'Date', 'Financial Period Absolute', 'Calc Date', 'ESG Score', 'Environmental Pillar Score', 'Social Pillar Score', 'Governance Pillar Score', 'ESG Combined Score Grade', 'year_esg', 'lag_year', 'type1_flag', 'type2_flag', 'type3_flag']


In [ ]:
ESG_COLS = [
    "Date", "Financial Period Absolute", "Calc Date",
    "ESG Score",
    "Environmental Pillar Score",
    "Social Pillar Score",
    "Governance Pillar Score",
    "ESG Combined Score Grade"
]

In [ ]:
def supplement_with_2014_esg(final_df, esg_2014_df):
    df = final_df.copy()

    # merge 2014 ESG
    merged = df.merge(
        esg_2014_df,
        left_on=["RIC", "year"],
        right_on=["RIC", "lag_year"],
        how="left",
        suffixes=("", "_2014")
    )

    # Only fill in the originally empty ESG fields
    for col in ESG_COLS:
        col_2014 = f"{col}_2014"
        if col in merged.columns and col_2014 in merged.columns:
            merged[col] = merged[col].combine_first(merged[col_2014])

    # If necessary, also add Financial Period Absolute / Date/Calc Date
    for extra_col in ["Date", "Financial Period Absolute", "Calc Date", "year", "lag_year"]:
        col_2014 = f"{extra_col}_2014"
        if extra_col in merged.columns and col_2014 in merged.columns:
            merged[extra_col] = merged[extra_col].combine_first(merged[col_2014])

    # Delete the _2014 field from merge
    drop_cols = [c for c in merged.columns if c.endswith("_2014")]
    merged = merged.drop(columns=drop_cols)

    return merged

spy_final_v2 = supplement_with_2014_esg(spy_final, spy_esg_2014_clean)
eem_final_v3 = supplement_with_2014_esg(eem_final, eem_esg_2014_clean)

print("SPY final v2 shape:", spy_final_v2.shape)
print("EEM final v3 shape:", eem_final_v3.shape)

print("\nSPY final v2 preview:")
print(spy_final_v2.head())

print("\nEEM final v3 preview:")
print(eem_final_v3.head())

SPY final v2 shape: (4005, 26)
EEM final v3 shape: (8017, 26)

SPY final v2 preview:
   etf        date        RIC                         Name        Country  \
0  SPY  2015-12-31      MMM.N                    3M CO ORD  UNITED STATES   
1  SPY  2015-12-31      ABT.N      ABBOTT LABORATORIES ORD  UNITED STATES   
2  SPY  2015-12-31     ABBV.N               ABBVIE INC ORD  UNITED STATES   
3  SPY  2015-12-31      ACN.N            ACCENTURE PLC ORD        IRELAND   
4  SPY  2015-12-31  ATVIn.BCU  ACTIVISION BLIZZARD INC ORD  UNITED STATES   

     Weight  No. Shares    old_RIC  year                Name_norm  ...  \
0  0.005164     6240677      MMM.N  2015                    3M CO  ...   
1  0.003731    15123927      ABT.N  2015      ABBOTT LABORATORIES  ...   
2  0.005392    16569595     ABBV.N  2015               ABBVIE INC  ...   
3  0.003636     6334755      ACN.N  2015            ACCENTURE PLC  ...   
4  0.001078     5071575  ATVIn.BCU  2015  ACTIVISION BLIZZARD INC  ...   

   ESG 

/tmp/ipykernel_32445/2376992995.py:23: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  merged[extra_col] = merged[extra_col].combine_first(merged[col_2014])
/tmp/ipykernel_32445/2376992995.py:23: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  merged[extra_col] = merged[extra_col].combine_first(merged[col_2014])


Paragraph 3: Check if the missing values ​​were successfully filled

It is recommended that you first check if the missing values ​​for 2015 have decreased, because the 2014 ESG was used to fill the missing values ​​for 2015.

In [ ]:
def check_esg_fill(df, label):
    print(f"\n=== {label} ===")
    for col in ESG_COLS:
        if col in df.columns:
            print(f"{col} missing rate: {df[col].isna().mean():.4f}")

    print("\n2015 sample preview:")
    cols_to_show = ["RIC", "year"] + [c for c in ESG_COLS if c in df.columns]
    print(df[df["year"] == 2015][cols_to_show].head(10))

check_esg_fill(spy_final_v2, "SPY final v2")
check_esg_fill(eem_final_v3, "EEM final v3")


=== SPY final v2 ===
Date missing rate: 0.0197
Financial Period Absolute missing rate: 0.0197
Calc Date missing rate: 0.0197
ESG Score missing rate: 0.0232
Environmental Pillar Score missing rate: 0.0232
Social Pillar Score missing rate: 0.0232
Governance Pillar Score missing rate: 0.0232
ESG Combined Score Grade missing rate: 0.0232

2015 sample preview:
         RIC  year       Date Financial Period Absolute            Calc Date  \
0      MMM.N  2015 2014-12-31                    FY2014  2015-12-31 00:00:00   
1      ABT.N  2015 2014-12-31                    FY2014  2015-12-31 00:00:00   
2     ABBV.N  2015 2014-12-31                    FY2014  2015-12-31 00:00:00   
3      ACN.N  2015 2014-08-31                    FY2014  2014-08-31 00:00:00   
4  ATVIn.BCU  2015 2014-12-31                    FY2014  2014-12-31 00:00:00   
5    ADBE.OQ  2015 2014-11-28                    FY2014  2015-11-27 00:00:00   
6      ADT.N  2015        NaT                       NaN                  NaT   
7

Paragraph 4: Save as a new file

In [ ]:
spy_final_v2.to_excel(f"{CLEAN_DIR}/SPY_final_cleaned_with_lag_esg_v2.xlsx", index=False)
eem_final_v3.to_excel(f"{CLEAN_DIR}/EEM_final_cleaned_with_lag_esg_v3.xlsx", index=False)

print(f"Saved: {CLEAN_DIR}/SPY_final_cleaned_with_lag_esg_v2.xlsx")
print(f"Saved: {CLEAN_DIR}/EEM_final_cleaned_with_lag_esg_v3.xlsx")

Saved: /content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/clean_data/SPY_final_cleaned_with_lag_esg_v2.xlsx
Saved: /content/drive/MyDrive/Colab Notebooks/Data Analytics/group project/clean_data/EEM_final_cleaned_with_lag_esg_v3.xlsx
